# Experiment 6: Comprehensive Caching Benchmark

## Objective

Evaluate the performance of different caching strategies using the complete SEVA_AI golden test dataset containing 973 questions.

We will compare uncached and cached RAG execution to determine the impact on latency, resource utilization, API costs and answer quality.

## Benchmark configurations

1. No caching (baseline).
2. Embedding caching.
3. Retrieval caching.
4. Parent document caching.
5. LLM response caching.
6. Combined caching.

## Evaluation metrics

- Average latency.
- P50, P95 and P99 latency.
- Cache hit rate and miss rate.
- Redis memory consumption.
- Embedding generation time.
- Retrieval time.
- LLM generation time.
- Token consumption and estimated API cost.
- Answer correctness and faithfulness.

## Dataset

Golden test dataset: 973 questions.

All benchmark configurations must use the same questions and consistent retrieval and generation settings.

## 1. Load Golden Test Dataset

Load the existing 973-question golden test dataset.

Purpose:
- Use identical queries across benchmark configurations.
- Preserve ground-truth answers and relevant scheme IDs.
- Establish a consistent evaluation dataset.
- Avoid changing the original golden dataset.

In [1]:
import json
import pandas as pd
from pathlib import Path

GOLDEN_TEST_PATH = Path(
    "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\final\\Final_dataset_splits\\test.json"
)

with open(GOLDEN_TEST_PATH, "r", encoding="utf-8") as f:
    golden_data = json.load(f)

golden_df = pd.DataFrame(golden_data)

print("Total test questions:", len(golden_df))
print("\nColumns:", golden_df.columns.tolist())
print("\nSample:")
display(golden_df.head(3))

Total test questions: 973

Columns: ['test_id', 'faq_id', 'scheme_id', 'scheme_name', 'question', 'ground_truth_answer', 'categories', 'level', 'source_url', 'relevant_scheme_ids', 'relevant_doc_ids', 'answerable']

Sample:


,test_id,faq_id,scheme_id,scheme_name,question,ground_truth_answer,categories,level,source_url,relevant_scheme_ids,relevant_doc_ids,answerable
0,GD_000031,0Be1F5gB5uF80_MGnI9U_FAQ_001,0Be1F5gB5uF80_MGnI9U,Dayanand Bandodkar Scheme For Higher Education...,Is this scheme limited to certain fields of st...,"The scheme covers a wide range of fields, incl...",[Education & Learning],State,https://www.myscheme.gov.in/schemes/dbsheo,[0Be1F5gB5uF80_MGnI9U],[],True
1,GD_004164,dxfCF5gB5uF80_MGAJJb_FAQ_006,dxfCF5gB5uF80_MGAJJb,PMMSY: Construction of Fish Retail Markets inc...,Is there a limit to how many units of central ...,"Yes, central assistance is restricted to one u...","[Agriculture,Rural & Environment, Business & E...",State,https://www.myscheme.gov.in/schemes/pmmsycrmhr,[dxfCF5gB5uF80_MGAJJb],[],True
2,GD_001489,FBfZF5gB5uF80_MGUpOf_FAQ_020,FBfZF5gB5uF80_MGUpOf,Accidental Benefit,What form needs to be used to apply for this w...,The application must be made in the prescribed...,[Social welfare & Empowerment],State,https://www.myscheme.gov.in/schemes/adwb,[FBfZF5gB5uF80_MGUpOf],[],True


In [2]:
required_columns = [
    "test_id",
    "question",
    "ground_truth_answer",
    "relevant_scheme_ids",
    "answerable"
]

missing_columns = [
    col for col in required_columns
    if col not in golden_df.columns
]

print("Missing columns:", missing_columns)
print("Total records:", len(golden_df))
print("Unique test IDs:", golden_df["test_id"].nunique())
print("Missing questions:", golden_df["question"].isna().sum())
print("Answerable questions:", golden_df["answerable"].sum())

Missing columns: []
Total records: 973
Unique test IDs: 973
Missing questions: 0
Answerable questions: 973


## 3. Baseline Retrieval Without Caching

Execute all 973 golden test questions directly against the existing Chroma discovery index.

**Configuration:**
- Vector database: Chroma
- Collection: seva_discovery_v1
- Candidate chunks: Top 30
- Unique scheme candidates: Top 10
- Redis caching: Disabled
- Embedding cache: Disabled

**Purpose:**
- Measure baseline retrieval latency.
- Identify whether the relevant scheme appears in retrieved results.
- Establish Recall@5, Recall@10, MRR and NDCG@10.
- Create a reference dataset for future caching comparisons.

We will aggregate retrieved chunks by scheme_id and preserve their original ranking.

In [3]:
import json
import time
from pathlib import Path
from collections import defaultdict

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [4]:
OUTPUT_DIR1 = Path("G:\\SEVA_Ai\\SEVA_Ai\\output\\SEVA_CHILDREN")

VECTORSTORE_DIR = OUTPUT_DIR1 / "vectorstores"

PARENT_PATH = OUTPUT_DIR1 / "parent_documents.jsonl"

print("Parent file exists:", PARENT_PATH.exists())
print("Vectorstore directory exists:", VECTORSTORE_DIR.exists())

Parent file exists: True
Vectorstore directory exists: True


In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [ ]:
discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)
detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


In [7]:
import time
import numpy as np
from collections import defaultdict

def baseline_retrieval(query, chunk_k=30, scheme_k=10):

    start = time.perf_counter()

    results = discovery_db.similarity_search_with_score(
        query,
        k=chunk_k
    )

    # Aggregate chunks by scheme ID
    scheme_scores = {}

    for doc, distance in results:
        scheme_id = doc.metadata.get("scheme_id")

        if scheme_id is None:
            continue

        if (
            scheme_id not in scheme_scores
            or distance < scheme_scores[scheme_id]
        ):
            scheme_scores[scheme_id] = distance

    # Rank unique schemes by minimum distance
    ranked_schemes = sorted(
        scheme_scores.items(),
        key=lambda x: x[1]
    )

    ranked_scheme_ids = [
        scheme_id
        for scheme_id, score in ranked_schemes[:scheme_k]
    ]

    latency = (time.perf_counter() - start) * 1000

    return {
        "ranked_scheme_ids": ranked_scheme_ids,
        "retrieved_chunks": len(results),
        "unique_schemes": len(scheme_scores),
        "latency_ms": latency
    }

## 4. Execute Baseline Retrieval on Golden Dataset

Run the uncached retrieval function for all 973 test questions.

**Important:**
- Use the same queries for every future benchmark configuration.
- Do not use Redis embedding or retrieval caches.
- Record individual query latency.
- Preserve retrieved scheme rankings for evaluation.

The output will be our baseline retrieval results.

In [8]:
baseline_results = []

for idx, row in golden_df.iterrows():

    result = baseline_retrieval(
        query=row["question"],
        chunk_k=30,
        scheme_k=10
    )

    baseline_results.append({
        "test_id": row["test_id"],
        "question": row["question"],
        "ground_truth_scheme_ids": row["relevant_scheme_ids"],
        **result
    })

    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/973 questions")

baseline_df = pd.DataFrame(baseline_results)

print("\nTotal processed:", len(baseline_df))
print("Average latency:", round(baseline_df["latency_ms"].mean(), 2), "ms")

Processed 100/973 questions
Processed 200/973 questions
Processed 300/973 questions
Processed 400/973 questions
Processed 500/973 questions
Processed 600/973 questions
Processed 700/973 questions
Processed 800/973 questions
Processed 900/973 questions

Total processed: 973
Average latency: 134.0 ms


## 5. Inspect Baseline Retrieval Results

Inspect the retrieved scheme IDs and latency for individual queries.

Verify that:
- All 973 questions were processed.
- Retrieved scheme IDs are populated.
- Retrieval latency is recorded.
- Ground-truth scheme IDs are preserved.

In [10]:
display(
    baseline_df[
        [
            "test_id",
            "retrieved_chunks",
            "unique_schemes",
            "ground_truth_scheme_ids",
            "ranked_scheme_ids",
            "latency_ms"
        ]
    ].head(10)
)

print("\nMissing retrieval results:",
      baseline_df["ranked_scheme_ids"].isna().sum())

print("Total retrieved chunks:",
      baseline_df["retrieved_chunks"].sum())

,test_id,retrieved_chunks,unique_schemes,ground_truth_scheme_ids,ranked_scheme_ids,latency_ms
0,GD_000031,30,27,[0Be1F5gB5uF80_MGnI9U],"[3ca-F5gB0AGc0F0iYTUA, cRhQHJ4B5uF80_MGdWqo, N...",1027.1135
1,GD_004164,30,30,[dxfCF5gB5uF80_MGAJJb],"[Zce6R5wB0AGc0F0ijAOI, AMa5F5gB0AGc0F0iwjVE, m...",461.5187
2,GD_001489,30,24,[FBfZF5gB5uF80_MGUpOf],"[Zca7F5gB0AGc0F0i0TVo, SBe7F5gB5uF80_MG1JE_, o...",501.5371
3,GD_002828,30,21,[RaE9GJgBXu-iUKXqCHkv],"[0xe-F5gB5uF80_MGZZFD, 36G_F5gBXu-iUKXqgXY5, g...",406.9232
4,GD_000477,30,30,[5MbVF5gB0AGc0F0isDb-],"[8KJMo58BXu-iUKXqflrJ, pcZFp5oB0AGc0F0iIaoe, C...",346.1603
5,GD_003934,30,26,[bKFTJJoBXu-iUKXqjHrW],"[SqE9GJgBXu-iUKXqCHmc, nBhD4JsB5uF80_MGDVsg, T...",467.4879
6,GD_002406,30,26,[NRezF5gB5uF80_MGGo-5],"[Sce1cp4B0AGc0F0inRMp, 8ccTkZ4B0AGc0F0i9hMs, A...",513.7423
7,GD_000169,30,19,[1selQJ8B0AGc0F0izxWD],"[QcZklJkB0AGc0F0igzlj, 3Be1F5gB5uF80_MGnY_A, 7...",516.6341
8,GD_006084,30,25,[wKL_z5oBXu-iUKXqtkCg],"[T6G3F5gBXu-iUKXqM3Uy, taG9F5gBXu-iUKXq4na1, z...",712.5943
9,GD_000536,30,23,[66I40ZwBXu-iUKXqEEk4],"[66I40ZwBXu-iUKXqEEk4, E6G_F5gBXu-iUKXq8nfc, D...",843.4181



Missing retrieval results: 0
Total retrieved chunks: 29190


## 6. Evaluate Baseline Retrieval Quality

Calculate retrieval quality using the complete golden dataset.

### Metrics

1. Recall@5: Percentage of relevant schemes retrieved in the top 5.
2. Recall@10: Percentage of relevant schemes retrieved in the top 10.
3. MRR@10: Measures the reciprocal rank of the first relevant scheme.
4. NDCG@10: Measures the ranking quality using graded or binary relevance.

For this experiment, use binary relevance based on `relevant_scheme_ids`.

These metrics establish the retrieval quality baseline before introducing caching.

## 6. Initialize Ragas and Langfuse Evaluation

Prepare the Ragas evaluation metrics and Langfuse experiment environment.

We will use Ragas for retrieval and answer quality evaluation, and Langfuse to store and compare experiment results.

The evaluation will use all 973 golden test questions.

In [9]:
import importlib.metadata as metadata

for package in ["ragas", "langfuse", "datasets"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: Not installed")

# Verify Ragas metric imports
from ragas.metrics import (
    IDBasedContextPrecision,
    IDBasedContextRecall
)

from ragas.dataset_schema import SingleTurnSample

print("\nRagas evaluation imports successful!")

ragas: 0.4.4
langfuse: 5.2.0
datasets: 5.2.0

Ragas evaluation imports successful!


C:\Users\devendra\AppData\Local\Temp\ipykernel_13204\2701990353.py:10: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (
C:\Users\devendra\AppData\Local\Temp\ipykernel_13204\2701990353.py:10: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (


## 6. Ragas-Based Retrieval Evaluation

Evaluate baseline retrieval quality using Ragas built-in ID-based metrics.

### Metrics

1. ID-Based Context Precision
2. ID-Based Context Recall

### Evaluation methodology

- Retrieved context IDs: Retrieved scheme IDs from baseline retrieval.
- Reference context IDs: Ground-truth relevant scheme IDs.
- Dataset: All 973 golden test questions.

These metrics evaluate scheme-level retrieval quality.

Note: This is scheme-level evaluation, not child-document-level evaluation. Our current golden dataset does not contain relevant document IDs.

In [10]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    IDBasedContextPrecision,
    IDBasedContextRecall
)

precision_metric = IDBasedContextPrecision()
recall_metric = IDBasedContextRecall()

print("Ragas retrieval evaluators initialized.")

Ragas retrieval evaluators initialized.


C:\Users\devendra\AppData\Local\Temp\ipykernel_13204\1149062708.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (
C:\Users\devendra\AppData\Local\Temp\ipykernel_13204\1149062708.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (


## 7. Execute Ragas Evaluation

Evaluate all retrieved scheme rankings against their ground-truth scheme IDs.

Ragas will calculate precision and recall for each test question.

We will preserve individual scores for analysis and comparison with future caching experiments.

In [12]:
def normalize_ids(value):
    if isinstance(value, list):
        return [str(x) for x in value]

    if isinstance(value, str):
        return json.loads(value) if value.startswith("[") else [value]

    return list(value) if value else []


async def run_ragas_retrieval_evaluation(baseline_df):

    results = []

    for idx, row in baseline_df.iterrows():

        retrieved_ids = normalize_ids(
            row["ranked_scheme_ids"]
        )

        reference_ids = normalize_ids(
            row["ground_truth_scheme_ids"]
        )

        sample = SingleTurnSample(
            retrieved_context_ids=retrieved_ids,
            reference_context_ids=reference_ids
        )

        precision = await precision_metric.single_turn_ascore(sample)
        recall = await recall_metric.single_turn_ascore(sample)

        results.append({
            "test_id": row["test_id"],
            "context_precision": float(precision),
            "context_recall": float(recall)
        })

        if (idx + 1) % 100 == 0:
            print(f"Evaluated {idx + 1}/973")

    return pd.DataFrame(results)


ragas_results = await run_ragas_retrieval_evaluation(
    baseline_df
)

print("\nEvaluation completed:", len(ragas_results))

Evaluated 100/973
Evaluated 200/973
Evaluated 300/973
Evaluated 400/973
Evaluated 500/973
Evaluated 600/973
Evaluated 700/973
Evaluated 800/973
Evaluated 900/973

Evaluation completed: 973


## 8. Baseline Ragas Evaluation Results

Aggregate the individual Ragas scores to establish the baseline retrieval quality.

These scores will be recorded in Langfuse for comparison against future retrieval and caching experiments.

In [13]:
ragas_baseline_scores = ragas_results[
    ["context_precision", "context_recall"]
].mean()

display(ragas_baseline_scores.to_frame("Ragas Score"))

,Ragas Score
context_precision,0.022325
context_recall,0.223022


In [14]:
golden_df["relevant_scheme_ids"] = golden_df["relevant_scheme_ids"].apply(
    normalize_ids
)

print("Relevant schemes per question:")
display(
    golden_df["relevant_scheme_ids"]
    .apply(len)
    .value_counts()
    .sort_index()
    .to_frame("question_count")
)

print("\nSample ground-truth labels:")
display(
    golden_df[
        ["test_id", "question", "relevant_scheme_ids"]
    ].head(5)
)

Relevant schemes per question:


,question_count
relevant_scheme_ids,
1,973



Sample ground-truth labels:


,test_id,question,relevant_scheme_ids
0,GD_000031,Is this scheme limited to certain fields of st...,[0Be1F5gB5uF80_MGnI9U]
1,GD_004164,Is there a limit to how many units of central ...,[dxfCF5gB5uF80_MGAJJb]
2,GD_001489,What form needs to be used to apply for this w...,[FBfZF5gB5uF80_MGUpOf]
3,GD_002828,What type of animals are covered under this sc...,[RaE9GJgBXu-iUKXqCHkv]
4,GD_000477,What documents are required to establish eligi...,[5MbVF5gB0AGc0F0isDb-]


In [15]:
depth_results = {}

for scheme_k in [3, 5, 10]:
    rows = []

    for _, row in golden_df.iterrows():
        result = baseline_retrieval(
            row["question"],
            chunk_k=30,
            scheme_k=scheme_k
        )

        rows.append({
            "test_id": row["test_id"],
            "ranked_scheme_ids": result["ranked_scheme_ids"],
            "ground_truth_scheme_ids": row["relevant_scheme_ids"]
        })

    experiment_df = pd.DataFrame(rows)

    evaluated = await run_ragas_retrieval_evaluation(experiment_df)

    depth_results[scheme_k] = evaluated[
        ["context_precision", "context_recall"]
    ].mean()

depth_comparison = pd.DataFrame(depth_results).T
depth_comparison.index.name = "scheme_k"

display(depth_comparison)


Evaluated 100/973
Evaluated 200/973
Evaluated 300/973
Evaluated 400/973
Evaluated 500/973
Evaluated 600/973
Evaluated 700/973
Evaluated 800/973
Evaluated 900/973
Evaluated 100/973
Evaluated 200/973
Evaluated 300/973
Evaluated 400/973
Evaluated 500/973
Evaluated 600/973
Evaluated 700/973
Evaluated 800/973
Evaluated 900/973
Evaluated 100/973
Evaluated 200/973
Evaluated 300/973
Evaluated 400/973
Evaluated 500/973
Evaluated 600/973
Evaluated 700/973
Evaluated 800/973
Evaluated 900/973


,context_precision,context_recall
scheme_k,,
3,0.055841,0.167523
5,0.036588,0.182939
10,0.022325,0.223022


In [18]:
golden_df["retrieval_query"] = golden_df.apply(
    lambda row: (
        f"For the scheme '{row['scheme_name']}': "
        f"{row['question']}"
    ),
    axis=1
)

display(
    golden_df[
        ["scheme_name", "question", "retrieval_query"]
    ].head(5)
)

,scheme_name,question,retrieval_query
0,Dayanand Bandodkar Scheme For Higher Education...,Is this scheme limited to certain fields of st...,For the scheme 'Dayanand Bandodkar Scheme For ...
1,PMMSY: Construction of Fish Retail Markets inc...,Is there a limit to how many units of central ...,For the scheme 'PMMSY: Construction of Fish Re...
2,Accidental Benefit,What form needs to be used to apply for this w...,For the scheme 'Accidental Benefit ': What for...
3,Establishment of 1 to 20 Milch Animal Farm,What type of animals are covered under this sc...,For the scheme 'Establishment of 1 to 20 Milch...
4,Delhi Arogya Nidhi,What documents are required to establish eligi...,For the scheme 'Delhi Arogya Nidhi': What docu...


3. Rerun the baseline with augmented queries
Markdown cell — Scheme-aware retrieval baseline

Use the same index, embedding model, chunking strategy, and retrieval parameters. Change only the query text.

In [19]:
scheme_aware_rows = []

for _, row in golden_df.iterrows():
    result = baseline_retrieval(
        row["retrieval_query"],
        chunk_k=30,
        scheme_k=10
    )

    scheme_aware_rows.append({
        "test_id": row["test_id"],
        "ranked_scheme_ids": result["ranked_scheme_ids"],
        "ground_truth_scheme_ids": row["relevant_scheme_ids"],
        "latency_ms": result["latency_ms"]
    })

scheme_aware_df = pd.DataFrame(scheme_aware_rows)

scheme_aware_results = await run_ragas_retrieval_evaluation(
    scheme_aware_df
)

display(
    scheme_aware_results[
        ["context_precision", "context_recall"]
    ].mean().to_frame("Scheme-aware Ragas Score")
)

Evaluated 100/973
Evaluated 200/973
Evaluated 300/973
Evaluated 400/973
Evaluated 500/973
Evaluated 600/973
Evaluated 700/973
Evaluated 800/973
Evaluated 900/973


,Scheme-aware Ragas Score
context_precision,0.099016
context_recall,0.986639


Scheme-aware top-1 retrieval

In [20]:
top1_rows = []

for _, row in golden_df.iterrows():
    result = baseline_retrieval(
        row["retrieval_query"],
        chunk_k=30,
        scheme_k=1
    )

    top1_rows.append({
        "test_id": row["test_id"],
        "ranked_scheme_ids": result["ranked_scheme_ids"],
        "ground_truth_scheme_ids": row["relevant_scheme_ids"]
    })

top1_df = pd.DataFrame(top1_rows)

top1_results = await run_ragas_retrieval_evaluation(top1_df)

display(
    top1_results[
        ["context_precision", "context_recall"]
    ].mean().to_frame("Top-1 Ragas Score")
)

Evaluated 100/973
Evaluated 200/973
Evaluated 300/973
Evaluated 400/973
Evaluated 500/973
Evaluated 600/973
Evaluated 700/973
Evaluated 800/973
Evaluated 900/973


,Top-1 Ragas Score
context_precision,0.922919
context_recall,0.922919


Top-1 retrieval error analysis

Identify queries where the first retrieved scheme differs from the ground-truth scheme. This will help us understand ranking failures

In [21]:
top1_error_df = top1_df[
    top1_df.apply(
        lambda row: (
            row["ranked_scheme_ids"][0]
            if row["ranked_scheme_ids"]
            else None
        ) not in row["ground_truth_scheme_ids"],
        axis=1
    )
].merge(
    golden_df[["test_id", "scheme_name", "question"]],
    on="test_id",
    how="left"
)

print("Top-1 failed questions:", len(top1_error_df))

display(
    top1_error_df[
        [
            "test_id",
            "scheme_name",
            "question",
            "ground_truth_scheme_ids",
            "ranked_scheme_ids"
        ]
    ].head(20)
)

Top-1 failed questions: 75


,test_id,scheme_name,question,ground_truth_scheme_ids,ranked_scheme_ids
0,GD_004164,PMMSY: Construction of Fish Retail Markets inc...,Is there a limit to how many units of central ...,[dxfCF5gB5uF80_MGAJJb],[kBcoqJgB5uF80_MGRJSb]
1,GD_002595,Top Working Program,What is the objective of the scheme?,[Phdsm5oB5uF80_MGHd8I],[c8aGMZgB0AGc0F0iGDiy]
2,GD_000111,Pradhan Mantri Matsya Sampada Yojana (PMMSY-In...,What is the minimum age requirement to be an a...,[1KFo_pgBXu-iUKXqN3lO],[kRjOZZsB5uF80_MG-ViC]
3,GD_004410,Scheme for Goa Rajya Vidnyanik Puraskar,Must applicants applying for this prestigious ...,[h6EsOpoBXu-iUKXqp3oT],[s6HTF5gBXu-iUKXqpXfs]
4,GD_004761,Namo Shetkari Mahasanman Nidhi Yojana,Is it mandatory to link bank account with Aadh...,[jRe5F5gB5uF80_MGSpCZ],[qxf9F5gB5uF80_MGZZOv]
5,GD_006381,Workers In Case Of Permanent Disablement-JKBOCWWB,What documents are required to apply for the s...,[z6G-F5gBXu-iUKXqYnaq],[6ca-F5gB0AGc0F0iYjWY]
6,GD_005891,Maternity Benefit to the Female Beneficiary,How many times can a worker avail of this bene...,[u6G-F5gBXu-iUKXqYHbl],[1Re-F5gB5uF80_MGaZEt]
7,GD_002869,Civil Service Incentive Scheme (Madhya Pradesh),Can I receive the incentive if I fail in any e...,[RcbbF5gB0AGc0F0i3zc0],[_Rc7GJgB5uF80_MGJ5PH]
8,GD_005655,Savitribai Phule Scholarship For V.J.N.T And S...,Is The Savitribai Phule Scholarship Available ...,[rRe1F5gB5uF80_MGIo-a],[3aG0F5gBXu-iUKXqKXR9]
9,GD_001412,Rise Together,What is the minimum age to apply under this sc...,[Esa1F5gB0AGc0F0iJTSh],[fxe9F5gB5uF80_MGZpEH]
